In [ ]:
!pip install langchain langchain_openai langsmith pandas langchain_experimental matplotlib langgraph langchain_core duckduckgo-search langchain-community chromadb

In [ ]:
import os
from uuid import uuid4

unique_id = uuid4().hex[0:8]
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"Read Text and DALL-E Image Drawing - {unique_id}"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = "여러분의_LANGCHAIN_API_KEY"
os.environ["OPENAI_API_KEY"] = ""

In [ ]:
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 Dall-E 모델을 위한 프롬프트를 생성하기 위한 임무를 가지고 있습니다."
            "제공된 텍스트를 토대로 Dall-E 모델이 이미지를 생성하기에 적절한 프롬프트를 생성하세요.",
        ),
        MessagesPlaceholder(variable_name="messages")
    ]
)

llm = ChatOpenAI(model="gpt-4o", temperature=0)
generate = llm | prompt

In [ ]:
news_text_sample_1 = """
인간은 다른 영장류에 비해 비교가 안 될 정도로 큰 뇌를 갖고 있다. 인간이 어떻게 비정상적으로 커진 뇌를 유지할 수 있는지 확인한 연구결과가 발표됐다. 뇌 질환 치료법을 찾는 데 도움이 될 것으로 보인다.

알렉스 폴른 미국 샌프란시스코캘리포니아대 신경학과 교수 연구팀은 인간의 뇌가 커다란 크기를 유지하기 위해 어떤 일을 하는지 살핀 연구결과를 논문 사전공개사이트 ‘바이오아카이브’에 15일 발표했다.

인간은 직립보행을 하는 과정에서 무릎과 허리 건강에 문제가 발생했다. 턱 구조 및 식단의 변화는 치열 불균형, 사랑니 매복 등의 문제를 일으켰다. 연구팀은 인간이 빠른 속도로 커진 뇌를 감당하기 위한 문제도 직면하고 있다고 강조했다.

연구팀은 움직임, 학습, 정서 처리에 중요한 신경전달물질인 ‘도파민’을 생성하는 신경세포에 주목했다. 인간, 침팬지, 원숭이, 오랑우탄에서 줄기세포를 채취한 뒤 실험실 환경에서 실제 뇌를 모방해 도파민을 생성하는 뇌 오가노이드(장기유사체)를 만들었다.

각 영장류의 뇌 오가노이드에서 도파민 신경세포를 비교한 결과 인간 도파민 신경세포는 다른 영장류에서보다 항산화 작용을 촉진하는 유전자가 더 많이 발현된다는 점이 확인됐다. 세포 손상의 일종인 산화 스트레스를 관리하는 유전자가 더 많이 만들어진다는 뜻이다.

큰 뇌를 유지하기 위한 에너지 집약 활동을 하려면 세포 손상이 많이 일어난다. 이를 감당하기 위해 인간의 뇌는 세포 손상을 관리하는 항산화 작용 유전자를 더 많이 만든다는 게 연구팀의 설명이다.

연구팀은 도파민이 분비되는 인간의 전전두엽 피질은 원숭이보다 18배, 선조체는 7배 크다는 점도 확인했다. 도파민 신경세포의 수는 다른 영장류의 약 2배에 불과했다. 인간의 도파민 신경세포는 멀리 뻗어 나가 부지런히 시냅스를 형성해나가야 한다는 의미다. 시냅스는 한 신경세포에서 다른 신경세포로 신호를 전달하는 연결 지점이다.

연구팀은 산화 스트레스를 유발하는 살충제를 뇌 오가노이드에 도포했을 때 인간의 신경세포에서 뇌유래신경영양인자(BDNF) 생성이 늘어난다는 점도 발견했다. BDNF는 파킨슨병과 같은 신경퇴행성장애가 있는 사람들에게서 감소한다. 연구팀은 “이번 연구는 파킨슨병, 조현병 등의 질환에 대한 이해를 높이는 데도 영감을 줄 것”이라고 말했다.
"""

In [ ]:
dalle_generation_prompt = ""
request = HumanMessage(
    content=news_text_sample_1
)
for chunk in generate.stream({"messages": [request]}):
    print(chunk.content, end="")
    dalle_generation_prompt += chunk.content

In [ ]:
from langchain.agents import initialize_agent, load_tools

tools = load_tools(["dalle-image-generator"])
agent = initialize_agent(tools, llm, agent="zero-shot-react-description", verbose=True)
output = agent.run(dalle_generation_prompt)

In [ ]:
from typing import Annotated, List, Sequence
from langgraph.graph import END, StateGraph, START
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

class State(TypedDict):
    messages: Annotated[list, add_messages]
    dalle_prompt: str

async def dalle_prompt_generation_node(state: State) -> State:
    return {"dalle_prompt": [await generate.invoke(state["messages"])]}

async def dalle_image_generation_node(state: State) -> State:
    dalle_generation_prompt = state["dalle_prompt"]

    tools = load_tools(["dalle-image-generator"])
    agent = initialize_agent(tools, llm, agent="zero-shot-react-description", verbose=True)
    output = agent.run(dalle_generation_prompt)

    return {"messages": [await generate.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("dalle_prompt_generate", dalle_prompt_generation_node)
builder.add_node("dalle_image_generator", dalle_image_generation_node)

builder.add_edge(START, "dalle_prompt_generator")
builder.add_edge('dalle_prompt_generator', "dalle_image_generate")
builder.add_edge("dalle_image_generate", END)

graph = builder.compile()